In [0]:
from pyspark.sql.functions import col, sum, when

df = spark.table("superstoreprac.silver.slv_table")
print(f"Total rows in Silver: {df.count()}")

Total rows in Silver: 9986


In [0]:
critical_columns = ['order_id', 'customer_id', 'product_id', 'sales', 'order_date']

null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in critical_columns
]).collect()[0].asDict()

print("Null counts in critical columns:")
for column, count in null_counts.items():
    print(f"  {column}: {count}")

failed_columns = [c for c, v in null_counts.items() if v > 0]

if failed_columns:
    raise Exception(f"❌ DATA QUALITY FAILURE: Nulls found in critical columns: {failed_columns}")
else:
    print("✅ PASSED: No nulls in critical columns")

Null counts in critical columns:
  order_id: 0
  customer_id: 0
  product_id: 0
  sales: 0
  order_date: 0
✅ PASSED: No nulls in critical columns


In [0]:
negative_sales_count = df.filter(col("sales") < 0).count()

print(f"Negative sales rows: {negative_sales_count}")

if negative_sales_count > 0:
    raise Exception(f"❌ DATA QUALITY FAILURE: {negative_sales_count} rows have negative sales")
else:
    print("✅ PASSED: No negative sales values")

Negative sales rows: 0
✅ PASSED: No negative sales values


In [0]:
invalid_discount_count = df.filter((col("discount") < 0) | (col("discount") > 1)).count()

print(f"Invalid discount rows: {invalid_discount_count}")

if invalid_discount_count > 0:
    raise Exception(f"❌ DATA QUALITY FAILURE: {invalid_discount_count} rows have discount outside 0-1 range")
else:
    print("✅ PASSED: All discount values within valid range")

Invalid discount rows: 0
✅ PASSED: All discount values within valid range


In [0]:
duplicate_count = (
    df.groupBy("order_id", "product_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

print(f"Duplicate order_id + product_id combinations: {duplicate_count}")

if duplicate_count > 0:
    raise Exception(f"❌ DATA QUALITY FAILURE: {duplicate_count} duplicate order-product combinations found")
else:
    print("✅ PASSED: No duplicates found")

Duplicate order_id + product_id combinations: 0
✅ PASSED: No duplicates found


In [0]:
row_count = df.count()
MIN_EXPECTED_ROWS = 9000
MAX_EXPECTED_ROWS = 11000

print(f"Current row count: {row_count}")

if row_count < MIN_EXPECTED_ROWS or row_count > MAX_EXPECTED_ROWS:
    raise Exception(f"❌ DATA QUALITY FAILURE: Row count {row_count} outside expected range [{MIN_EXPECTED_ROWS}-{MAX_EXPECTED_ROWS}]")
else:
    print("✅ PASSED: Row count within expected range")

Current row count: 9986
✅ PASSED: Row count within expected range


In [0]:
print("=" * 50)
print("✅ ALL DATA QUALITY CHECKS PASSED")
print("=" * 50)
print("Silver layer is verified clean. Proceeding to Gold layer.")

✅ ALL DATA QUALITY CHECKS PASSED
Silver layer is verified clean. Proceeding to Gold layer.
